# HMapp algorithme

This notebook attempts to reproduce the findings of the 2026 paper, more specifically the pipeline used to find these results. A few modifications will be made:
 - Use a already selected corpus of texts, thus skipping entirely the seed search as performed by the HMapp
 - Limited result testing: for now, lack of viable dataset

In [ ]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import bigrams

## Load the dataset

In [4]:
# load the data
df = pd.read_csv("./data/Unmasking Antisemitism SRI Data Set - Reporting Layer.csv")

In [5]:
df.describe()



,Time,Likes,Shares,Hateful,Violent,Offensive,Negative,Homophobic,Transphobic,Racist,Misogynistic,Xenophobic
count,0.0,295.000000,123.000000,659.000000,659.000000,659.000000,659.000000,660.000000,660.000000,660.000000,660.000000,660.000000
mean,NaN,11.301695,3.186992,0.335102,0.342416,0.491892,0.817020,0.075758,0.063636,0.227273,0.024242,0.096970
std,NaN,54.013797,16.545689,0.122211,0.312372,0.074944,0.265517,0.990250,0.834869,2.932190,0.329648,1.262145
min,NaN,0.000000,0.000000,0.159000,0.022000,0.397000,0.027000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.000000,0.250000,0.108000,0.458500,0.839500,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,1.000000,0.000000,0.309000,0.199000,0.479000,0.935000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,4.000000,1.000000,0.380500,0.515500,0.506000,0.967000,0.000000,0.000000,0.000000,0.000000,0.000000
max,NaN,584.000000,174.000000,0.859000,0.971000,0.991000,0.991000,25.000000,21.000000,75.000000,8.000000,32.000000


In [6]:
df.shape

(667, 29)

In [7]:
df.head()

,Term or Phrase,Date,Time,User,User Affiliation,Media Platform,Link to Post,Post Text,Screenshot,Image,...,Homophobic,Transphobic,Racist,Misogynistic,Xenophobic,Trope Classification,Primary Coder,Coder 2,Coder 3,Notes
0,Cabal,12/06/22,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@GlyMor/10946663297134...,"Exactly!!!! Bring in the mighty Marines, round...",Cabal_01,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,NaN
1,Cabal,1/19/2023,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@17QRanches/1097135105...,WINNING üí•WINNING üí•WINNING üí•CLEANING U...,Cabal_02,https://static-assets-1.truthsocial.com/tmtg:p...,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,Target is RHINOS
2,Cabal,12/27/2022,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@1776WeThePeople1776/1...,Mr President this Cabal must all be destroyed!...,Cabal_03,https://static-assets-1.truthsocial.com/tmtg:p...,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,NaN
3,Cabal,12/06/22,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@blakecrosby/109466136...,ITS VAX GENOCIDE A COLOR REVOLUTION AND THE CA...,Cabal_04,NaN,...,0.0,0.0,0.0,0.0,0.0,"1,8",coder 1,coder 2,coder 3,Jews as servants of Satan
4,Cabal,12/14/2022,NaN,Anonymous,NaN,Truth Social,https://truthsocial.com/@QuirkyOtter/109513112...,"Yes, Because CHILD PORNOGRAPHY Global Cabal! R...",Cabal_05,NaN,...,0.0,0.0,0.0,0.0,0.0,NaN,coder 1,coder 2,coder 3,NaN


## Candidate extraction

Now that are corpus is available to us, we can extract candidate terms as described

### Text pre-processing
This step includes removing hyperlinks, stop words, setting everything in lowercase, and constructing unigrams, bigrams and trigrams.

In [26]:
# copy our df
data = df.copy()
### cleaning ####
# remove hyperlinks
data['Post Text'] = data['Post Text'].str.replace(r'http\S+|www\S+|https\S+', '', regex=True, case=False)

# lowercase entire text
data['Post Text'] = data['Post Text'].str.lower()

# remove stopwords
# first, load stop words
nltk.download("stopwords")
nltk.download("punkt_tab") # have to download this resource as well for tokenization; not sure why

stop_words = set(stopwords.words('english'))

# remove stop words from the text
# use a lambda function for ease
# gets a type error, gets float instead of string, dont know why??
# isntead lets do a function
def remove_stopwords(text):
    tokens = word_tokenize(text)
    return ' '.join([word for word in tokens if word not in stop_words])

for idx, text in data["Post Text"].items():
    if isinstance(text, str):
        data.loc[idx, "Post Text"] = remove_stopwords(text)


print(f"Before stop word removal: \n{df['Post Text'].head()}")
print(f"After stop word removal: \n{data['Post Text'].head()}")

[nltk_data] Downloading package stopwords to /home/armand/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/armand/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Before stop word removal: 
0    Exactly!!!! Bring in the mighty Marines, round...
1    WINNING üí•WINNING üí•WINNING üí•CLEANING U...
2    Mr President this Cabal must all be destroyed!...
3    ITS VAX GENOCIDE A COLOR REVOLUTION AND THE CA...
4    Yes, Because CHILD PORNOGRAPHY Global Cabal! R...
Name: Post Text, dtype: str
After stop word removal: 
0    exactly ! ! ! ! bring mighty marines , round p...
1    winning üí•winning üí•winning üí•cleaning r...
2    mr president cabal must destroyed ! ! period !...
3    vax genocide color revolution cabal laughing w...
4    yes , child pornography global cabal ! right f...
Name: Post Text, dtype: str


In [ ]:
# construct unigram, bigram and trigram features
# def: a n-gram is a sequence of n adjacent words

